# 02d — Arena Log

Procesa `arena_log.csv` para generar features de PvP por usuario.

**Hallazgo crítico:** los campos `attacker_id` y `defender_id` contienen
CHARACTER_IDs, no user_ids. Se construye un mapeo desde `characters.csv`
para traducirlos antes de filtrar.

**Inputs:**
- `data/data_raw/arena_log.csv` (249K filas)
- `data/data_raw/characters.csv` (para mapeo char_id → user_id)
- `data/data_qc/sample_user_ids.parquet`

**Outputs:**
- `data/data_qc/features_arena.parquet` (1 fila/usuario)

**Complejidad especial:** cada combate involucra dos jugadores (attacker + defender).
Se normaliza a la perspectiva del usuario antes de agregar.

In [1]:
# [SETUP] Imports y dependencias
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gc
import time
from pathlib import Path
from datetime import datetime

plt.style.use('default')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

In [2]:
# [SETUP] Paths, constantes y helpers
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase1_cleaning' else Path.cwd()
DATA_RAW = PROJECT_ROOT / 'data' / 'data_raw'
DATA_QC = PROJECT_ROOT / 'data' / 'data_qc'
REPORT_DIR = PROJECT_ROOT / 'informes' / 'fase1_cleaning' / 'arena_log'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

CSV_INPUT = DATA_RAW / 'arena_log.csv'
CHARACTERS_CSV = DATA_RAW / 'characters.csv'
SAMPLE_IDS_PATH = DATA_QC / 'sample_user_ids_cutoff.parquet'
PARQUET_FEATURES = DATA_QC / 'features_arena_cutoff.parquet'

def clean_user_id(uid):
    uid = str(uid)
    if uid.startswith('ObjectId(') and uid.endswith(')'):
        return uid[9:-1].replace("'", "")
    return uid

steps_log = []
NOTEBOOK_START = time.time()

def log_step(tag, message):
    ts = datetime.now().strftime('%H:%M:%S')
    entry = f"[{tag}] {ts} — {message}"
    steps_log.append(entry)
    print(entry)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"CSV_INPUT existe: {CSV_INPUT.exists()}")
print(f"CHARACTERS_CSV existe: {CHARACTERS_CSV.exists()}")

PROJECT_ROOT: /Users/jezquerro/Documents/tfg
CSV_INPUT existe: True
CHARACTERS_CSV existe: True


In [3]:
# [SETUP] Carga de sample_user_ids y REFERENCE_DATE
sample_ids_df = pd.read_parquet(SAMPLE_IDS_PATH)
sample_ids_df['user_id'] = sample_ids_df['user_id'].apply(clean_user_id)
sample_user_ids = set(sample_ids_df['user_id'])
N_SAMPLE = len(sample_user_ids)

sample_example = list(sample_user_ids)[0]
assert len(sample_example) == 24 and not sample_example.startswith('ObjectId'), \
    f"ERROR: user_id no es hex limpio: '{sample_example}'"

users_clean = pd.read_parquet(DATA_QC / 'users_clean_cutoff.parquet', columns=['last_login_dt'])
REFERENCE_DATE = users_clean['last_login_dt'].max()
CUTOFF_DATE = REFERENCE_DATE - pd.Timedelta(days=120)
del users_clean
gc.collect()

print(f"Usuarios en sample: {N_SAMPLE:,}")
print(f"REFERENCE_DATE: {REFERENCE_DATE}")
log_step('SETUP', f'sample_user_ids: {N_SAMPLE:,} usuarios')
log_step('SETUP', f'REFERENCE_DATE: {REFERENCE_DATE}')

Usuarios en sample: 25,200
REFERENCE_DATE: 2026-04-04 18:23:13
[SETUP] 17:01:48 — sample_user_ids: 25,200 usuarios
[SETUP] 17:01:48 — REFERENCE_DATE: 2026-04-04 18:23:13


In [4]:
# [SETUP] Construir mapeo character_id → user_id desde characters.csv
# Necesario porque arena_log usa character_ids, no user_ids
t0 = time.time()

chars_df = pd.read_csv(CHARACTERS_CSV, usecols=['_id', 'user_id', 'created_at'], low_memory=False)
chars_df['created_at'] = pd.to_datetime(chars_df['created_at'], errors='coerce', utc=True)

# Filtro cutoff: ajustar tz (chars_df['created_at'] es tz-aware UTC, CUTOFF_DATE puede ser naive)
_col_tz = chars_df['created_at'].dt.tz
if _col_tz is None:
    _cut_aligned = CUTOFF_DATE.tz_localize(None) if CUTOFF_DATE.tz is not None else CUTOFF_DATE
else:
    _cut_aligned = CUTOFF_DATE if CUTOFF_DATE.tz is not None else CUTOFF_DATE.tz_localize('UTC')
n_pre_cut = len(chars_df)
chars_df = chars_df[chars_df['created_at'].notna() & (chars_df['created_at'] <= _cut_aligned)].copy()
log_step('SETUP', f'char_to_user filtrado por cutoff: {n_pre_cut:,} → {len(chars_df):,}')

# Extraer hex limpio del _id del personaje
chars_df['char_id'] = chars_df['_id'].str.extract(r"ObjectId\(?'?([a-f0-9]+)'?\)?")

# Crear mapeo char_id → user_id (solo personajes con user_id no nulo)
char_to_user = chars_df.dropna(subset=['user_id', 'char_id']).set_index('char_id')['user_id'].to_dict()

del chars_df
gc.collect()

print(f"Mapeo char_id → user_id construido: {len(char_to_user):,} entradas")
print(f"Tiempo: {time.time()-t0:.1f}s")
log_step('SETUP', f'Mapeo char→user: {len(char_to_user):,} entradas')

[SETUP] 17:01:51 — char_to_user filtrado por cutoff: 1,336,143 → 880,095


Mapeo char_id → user_id construido: 880,095 entradas
Tiempo: 5.0s
[SETUP] 17:01:53 — Mapeo char→user: 880,095 entradas


## 1. Carga y exploración del CSV

In [5]:
# [EXEC] Carga de arena_log.csv
t0 = time.time()
df_raw = pd.read_csv(CSV_INPUT, low_memory=False)
load_time = time.time() - t0

print(f"Shape: {df_raw.shape}")
print(f"Tiempo: {load_time:.1f}s")
print(f"Memoria: {df_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB")
log_step('EXEC', f'Carga CSV: shape={df_raw.shape}, tiempo={load_time:.1f}s')

Shape: (249583, 30)
Tiempo: 0.9s
Memoria: 100.0 MB
[EXEC] 17:01:54 — Carga CSV: shape=(249583, 30), tiempo=0.9s


In [6]:
# [ANALYSIS] Tipos de datos del CSV crudo
print("TIPOS DE DATOS — CSV CRUDO")
print("=" * 80)
for col in df_raw.columns:
    dtype = df_raw[col].dtype
    n_nulls = df_raw[col].isnull().sum()
    pct_nulls = n_nulls / len(df_raw) * 100
    n_unique = df_raw[col].nunique()
    example = df_raw[col].dropna().iloc[0] if n_nulls < len(df_raw) else 'TODO NULO'
    print(f"  {col:<30} dtype={str(dtype):<10} nulls={n_nulls:>8,} ({pct_nulls:5.1f}%)  "
          f"unique={n_unique:>8,}  ej='{str(example)[:35]}'")

TIPOS DE DATOS — CSV CRUDO
  _id                            dtype=str        nulls=       0 (  0.0%)  unique= 249,583  ej='ObjectId(69a8b97fdcfe0b3b527ae5ac)'
  time_attacked                  dtype=int64      nulls=       0 (  0.0%)  unique= 237,743  ej='1772665215'
  arena_category                 dtype=int64      nulls=       0 (  0.0%)  unique=      24  ej='14'
  is_arena_boss                  dtype=bool       nulls=       0 (  0.0%)  unique=       2  ej='False'
  attacker_id                    dtype=str        nulls=       0 (  0.0%)  unique=  10,053  ej='6977dcc085272d7d454ac59f'
  defender_id                    dtype=str        nulls=       0 (  0.0%)  unique=   5,101  ej='67eb7f27a1e7a43bee69f3b0'
  winner_id                      dtype=str        nulls=       0 (  0.0%)  unique=  12,034  ej='6977dcc085272d7d454ac59f'
  experience_reward              dtype=float64    nulls=  23,474 (  9.4%)  unique=     350  ej='788.0'
  gold_reward                    dtype=float64    nulls=  23,

In [7]:
# [ANALYSIS] Nulos por columna
nulls = df_raw.isnull().sum().to_frame(name='n_nulls')
nulls['pct_nulls'] = (nulls['n_nulls'] / len(df_raw) * 100).round(2)
nulls = nulls.sort_values('pct_nulls', ascending=False)
print(nulls[nulls['n_nulls'] > 0])
nulls.to_csv(REPORT_DIR / 'nulos_por_columna_raw.csv')
log_step('ANALYSIS', 'Nulos por columna guardados')

                   n_nulls  pct_nulls
experience_reward    23474       9.41
gold_reward          23474       9.41
dark_steel_reward    23474       9.41
[ANALYSIS] 17:01:54 — Nulos por columna guardados


In [8]:
# [ANALYSIS] Distribuciones clave del CSV crudo
print("=== is_arena_boss ===")
print(df_raw['is_arena_boss'].value_counts())

print("\n=== arena_category (distribución) ===")
print(df_raw['arena_category'].value_counts().sort_index())

print("\n=== Combates por atacante (top stats) ===")
att_counts = df_raw['attacker_id'].value_counts()
print(f"  Atacantes únicos: {att_counts.shape[0]:,}")
print(f"  Mediana combates: {att_counts.median():.0f}")
print(f"  Max combates: {att_counts.max()}")

print("\n=== Combates por defensor (top stats) ===")
def_counts = df_raw['defender_id'].value_counts()
print(f"  Defensores únicos: {def_counts.shape[0]:,}")
print(f"  Mediana combates: {def_counts.median():.0f}")
print(f"  Max combates: {def_counts.max()}")

# Verificar que los IDs son character_ids, no user_ids
all_ids = set(df_raw['attacker_id'].dropna()) | set(df_raw['defender_id'].dropna())
overlap_sample = len(all_ids & sample_user_ids)
overlap_chars = len(all_ids & set(char_to_user.keys()))
print(f"\n=== Verificación de tipo de ID ===")
print(f"  IDs únicos en arena: {len(all_ids):,}")
print(f"  Overlap con user_ids (sample): {overlap_sample} → {'son user_ids!' if overlap_sample > 100 else 'NO son user_ids'}")
print(f"  Overlap con char_ids (mapeo):  {overlap_chars:,} → {'son character_ids' if overlap_chars > 100 else 'NO son char_ids!'}")

log_step('ANALYSIS', f'Boss fights: {df_raw["is_arena_boss"].sum():,}, '
         f'IDs verificados como character_ids (overlap={overlap_chars:,})')

=== is_arena_boss ===
is_arena_boss
False    238262
True      11321
Name: count, dtype: int64

=== arena_category (distribución) ===
arena_category
0       516
1      2455
2       946
3      1769
4      3716
5      9273
6     12522
7     11538
8     13848
9     20981
10    21064
11    18613
12    22526
13    24023
14    22617
15    20744
16    12140
17    15319
18     4300
19     4661
20     1077
21      738
22      134
23     4063
Name: count, dtype: int64

=== Combates por atacante (top stats) ===
  Atacantes únicos: 10,053
  Mediana combates: 5
  Max combates: 3484

=== Combates por defensor (top stats) ===
  Defensores únicos: 5,101
  Mediana combates: 28
  Max combates: 1323



=== Verificación de tipo de ID ===
  IDs únicos en arena: 14,245
  Overlap con user_ids (sample): 0 → NO son user_ids
  Overlap con char_ids (mapeo):  5,368 → son character_ids
[ANALYSIS] 17:01:54 — Boss fights: 11,321, IDs verificados como character_ids (overlap=5,368)


## 2. Traducción de character_ids a user_ids y filtrado

Los campos `attacker_id` y `defender_id` contienen character_ids
(ObjectId del personaje), no user_ids. Usamos el mapeo construido
desde `characters.csv` para traducirlos.

Después filtramos por `sample_user_ids.parquet`.

In [9]:
# [EXEC] Traducir character_ids → user_ids y filtrar
t0 = time.time()
n_before = len(df_raw)

# Traducir attacker_id y defender_id a user_ids
df_raw['attacker_user_id'] = df_raw['attacker_id'].map(char_to_user)
df_raw['defender_user_id'] = df_raw['defender_id'].map(char_to_user)

# Estadísticas de traducción
n_att_translated = df_raw['attacker_user_id'].notna().sum()
n_def_translated = df_raw['defender_user_id'].notna().sum()
print(f"Atacantes traducidos: {n_att_translated:,}/{n_before:,} ({n_att_translated/n_before*100:.1f}%)")
print(f"Defensores traducidos: {n_def_translated:,}/{n_before:,} ({n_def_translated/n_before*100:.1f}%)")

# También traducir winner_id para referencia (pero is_win se calcula con char_ids originales)
df_raw['winner_user_id'] = df_raw['winner_id'].map(char_to_user)

# Filtrar: conservar combates donde AL MENOS uno de los dos está en sample
mask_att = df_raw['attacker_user_id'].isin(sample_user_ids)
mask_def = df_raw['defender_user_id'].isin(sample_user_ids)
df_filtered = df_raw[mask_att | mask_def].copy()
n_after = len(df_filtered)

print(f"\nCombates iniciales:              {n_before:>12,}")
print(f"Combates con jugador del sample: {n_after:>12,}  ({n_after/n_before*100:.1f}%)")
print(f"Tiempo: {time.time()-t0:.1f}s")

log_step('EXEC', f'Traducción + filtrado: {n_before:,} → {n_after:,} ({n_after/n_before*100:.1f}%)')

del df_raw
gc.collect()

Atacantes traducidos: 93,235/249,583 (37.4%)
Defensores traducidos: 212,433/249,583 (85.1%)



Combates iniciales:                   249,583
Combates con jugador del sample:      116,933  (46.9%)
Tiempo: 0.9s
[EXEC] 17:01:55 — Traducción + filtrado: 249,583 → 116,933 (46.9%)


0

In [10]:
# [EXEC] Parsear timestamp y normalizar a perspectiva del usuario
t0 = time.time()

df_filtered['fight_dt'] = pd.to_datetime(df_filtered['time_attacked'], unit='s', errors='coerce')

# Filtro cutoff: ajustar tz para que coincida con la columna
_col_tz = df_filtered['fight_dt'].dt.tz
if _col_tz is None:
    _cut_aligned = CUTOFF_DATE.tz_localize(None) if CUTOFF_DATE.tz is not None else CUTOFF_DATE
else:
    _cut_aligned = CUTOFF_DATE if CUTOFF_DATE.tz is not None else CUTOFF_DATE.tz_localize('UTC')
n_pre_cut = len(df_filtered)
df_filtered = df_filtered[df_filtered['fight_dt'].notna() & (df_filtered['fight_dt'] <= _cut_aligned)].copy()
log_step('EXEC', f'Filtro cutoff (fight_dt <= CUTOFF): {n_pre_cut:,} → {len(df_filtered):,}')

# --- Perspectiva ATACANTE ---
df_att = df_filtered[df_filtered['attacker_user_id'].isin(sample_user_ids)].copy()
df_att = df_att.rename(columns={
    'attacker_user_id': 'user_id',
    'attacker_trophies_reward': 'trophies_change',
    'attacker_final_trophies': 'final_trophies',
    'is_attacker_subscriber': 'is_subscriber',
    'attacker_level': 'user_level',
    'attacker_rank': 'user_rank',
    'defender_level': 'opponent_level',
})
df_att['role'] = 'attacker'
# is_win: comparar attacker_id (char_id original) con winner_id (también char_id)
df_att['is_win'] = (df_att['attacker_id'] == df_att['winner_id']).astype(int)

# --- Perspectiva DEFENSOR ---
df_def = df_filtered[df_filtered['defender_user_id'].isin(sample_user_ids)].copy()
df_def = df_def.rename(columns={
    'defender_user_id': 'user_id',
    'defender_trophies_lost': 'trophies_change',
    'defender_final_trophies': 'final_trophies',
    'is_defender_subscriber': 'is_subscriber',
    'defender_level': 'user_level',
    'defender_rank': 'user_rank',
    'attacker_level': 'opponent_level',
})
df_def['trophies_change'] = -df_def['trophies_change'].abs()
df_def['role'] = 'defender'
# is_win: comparar defender_id (char_id original) con winner_id (también char_id)
df_def['is_win'] = (df_def['defender_id'] == df_def['winner_id']).astype(int)
df_def['experience_reward'] = 0.0
df_def['gold_reward'] = 0.0
df_def['dark_steel_reward'] = 0.0

common_cols = ['user_id', 'fight_dt', 'arena_category', 'is_arena_boss',
               'experience_reward', 'gold_reward', 'dark_steel_reward',
               'trophies_change', 'final_trophies', 'is_subscriber',
               'user_level', 'user_rank', 'opponent_level', 'role', 'is_win']

df_normalized = pd.concat([df_att[common_cols], df_def[common_cols]], ignore_index=True)

n_users_arena = df_normalized['user_id'].nunique()
n_users_no_arena = N_SAMPLE - n_users_arena

print(f"Filas normalizadas: {len(df_normalized):,}")
print(f"  Como atacante: {len(df_att):,}")
print(f"  Como defensor: {len(df_def):,}")
print(f"\nUsuarios CON arena: {n_users_arena:,} ({n_users_arena/N_SAMPLE*100:.1f}%)")
print(f"Usuarios SIN arena: {n_users_no_arena:,} ({n_users_no_arena/N_SAMPLE*100:.1f}%)")
print(f"Tiempo: {time.time()-t0:.1f}s")

log_step('EXEC', f'Normalizado: {len(df_normalized):,} filas, {n_users_arena:,} usuarios con arena')

del df_filtered, df_att, df_def
gc.collect()

[EXEC] 17:01:55 — Filtro cutoff (fight_dt <= CUTOFF): 116,933 → 0
Filas normalizadas: 0
  Como atacante: 0
  Como defensor: 0

Usuarios CON arena: 0 (0.0%)
Usuarios SIN arena: 25,200 (100.0%)
Tiempo: 0.1s
[EXEC] 17:01:55 — Normalizado: 0 filas, 0 usuarios con arena


0

In [11]:
# [ANALYSIS] Estadísticas post-normalización
fights_per_user = df_normalized.groupby('user_id').size()
print(f"Combates por usuario:")
print(f"  Mediana: {fights_per_user.median():.0f}")
print(f"  Media: {fights_per_user.mean():.1f}")
print(f"  Max: {fights_per_user.max()}")
print(f"  P90: {fights_per_user.quantile(0.9):.0f}")
print(f"  P99: {fights_per_user.quantile(0.99):.0f}")

print(f"\nDistribución de roles:")
print(df_normalized['role'].value_counts())

print(f"\nWin rate global: {df_normalized['is_win'].mean():.3f}")

print(f"\nTipos de datos:")
for col in df_normalized.columns:
    print(f"  {col:<25} {str(df_normalized[col].dtype):<15}")

log_step('ANALYSIS', f'Stats: {len(df_normalized):,} filas, fights/user mediana={fights_per_user.median():.0f}')

Combates por usuario:
  Mediana: nan
  Media: nan
  Max: nan
  P90: nan
  P99: nan

Distribución de roles:
Series([], Name: count, dtype: int64)

Win rate global: nan

Tipos de datos:
  user_id                   str            
  fight_dt                  datetime64[s]  
  arena_category            int64          
  is_arena_boss             bool           
  experience_reward         float64        
  gold_reward               float64        
  dark_steel_reward         float64        
  trophies_change           int64          
  final_trophies            int64          
  is_subscriber             bool           
  user_level                int64          
  user_rank                 int64          
  opponent_level            int64          
  role                      str            
  is_win                    int64          
[ANALYSIS] 17:01:55 — Stats: 0 filas, fights/user mediana=nan


## 3. Agregación por usuario — ~21 features en 7 grupos

In [12]:
# [EXEC] Grupo A: Volumen de combates (3 features)
t0 = time.time()

group_a = df_normalized.groupby('user_id').agg(
    arena_fights_total=('is_win', 'size'),
)

role_counts = df_normalized.groupby(['user_id', 'role']).size().unstack(fill_value=0)
group_a['arena_fights_as_attacker'] = role_counts.get('attacker', 0)
group_a['arena_fights_as_defender'] = role_counts.get('defender', 0)

print(f"Grupo A: {len(group_a):,} filas, {group_a.shape[1]} features, {time.time()-t0:.1f}s")
log_step('EXEC', f'Grupo A (volumen): {group_a.shape[1]} features')

Grupo A: 0 filas, 3 features, 0.0s
[EXEC] 17:01:55 — Grupo A (volumen): 3 features


In [13]:
# [EXEC] Grupo B: Rendimiento (4 features)
t0 = time.time()

wins = df_normalized.groupby('user_id')['is_win'].sum().rename('arena_wins_total')
total = df_normalized.groupby('user_id')['is_win'].count()

group_b = pd.DataFrame({
    'arena_wins_total': wins,
    'arena_losses_total': total - wins,
    'arena_win_rate': (wins / total).round(3),
})

att_only = df_normalized[df_normalized['role'] == 'attacker']
att_wr = att_only.groupby('user_id')['is_win'].mean().rename('arena_win_rate_as_attacker').round(3)
group_b = group_b.join(att_wr)

print(f"Grupo B: {len(group_b):,} filas, {group_b.shape[1]} features, {time.time()-t0:.1f}s")
log_step('EXEC', f'Grupo B (rendimiento): {group_b.shape[1]} features')

Grupo B: 0 filas, 4 features, 0.0s
[EXEC] 17:01:55 — Grupo B (rendimiento): 4 features


In [14]:
# [EXEC] Grupo C: Trofeos (4 features — Fix B1 aplicado)
# Fix B1: arena_rank_best excluye rank=0 ("sin ranking asignado")
# y se añade arena_has_rank como flag.
# Ver informes/fase1_cleaning/arena_log/verificacion_02d.md
t0 = time.time()

trophies_max = df_normalized.groupby('user_id')['final_trophies'].max().rename('arena_trophies_max')
trophies_net = df_normalized.groupby('user_id')['trophies_change'].sum().rename('arena_trophies_net')

# Fix B1: calcular min(rank) SOLO sobre filas con rank > 0
df_with_rank = df_normalized[df_normalized['user_rank'] > 0]
rank_best = df_with_rank.groupby('user_id')['user_rank'].min().rename('arena_rank_best')

group_c = pd.concat([trophies_max, trophies_net, rank_best], axis=1)

print(f"Grupo C: {len(group_c):,} filas, {group_c.shape[1]} features, {time.time()-t0:.1f}s")
print(f"  Fix B1: usuarios con rank válido (rank>0): {rank_best.notna().sum():,}")
log_step('EXEC', f'Grupo C (trofeos): {group_c.shape[1]} features — Fix B1 aplicado')

Grupo C: 0 filas, 3 features, 0.0s
  Fix B1: usuarios con rank válido (rank>0): 0
[EXEC] 17:01:55 — Grupo C (trofeos): 3 features — Fix B1 aplicado


In [15]:
# [EXEC] Grupo D: Recompensas (3 features)
t0 = time.time()

group_d = df_normalized.groupby('user_id').agg(
    arena_gold_earned=('gold_reward', 'sum'),
    arena_exp_earned=('experience_reward', 'sum'),
    arena_dark_steel_earned=('dark_steel_reward', 'sum'),
)

print(f"Grupo D: {len(group_d):,} filas, {group_d.shape[1]} features, {time.time()-t0:.1f}s")
log_step('EXEC', f'Grupo D (recompensas): {group_d.shape[1]} features')

Grupo D: 0 filas, 3 features, 0.0s
[EXEC] 17:01:55 — Grupo D (recompensas): 3 features


In [16]:
# [EXEC] Grupo E: Competitividad y matchmaking (3 features)
t0 = time.time()

cat_max = df_normalized.groupby('user_id')['arena_category'].max().rename('arena_category_max')

df_normalized['level_diff'] = df_normalized['user_level'] - df_normalized['opponent_level']
avg_diff = df_normalized.groupby('user_id')['level_diff'].mean().rename('arena_avg_level_diff').round(2)

is_sub = df_normalized.groupby('user_id')['is_subscriber'].max().rename('arena_is_subscriber').astype(int)

group_e = pd.concat([cat_max, avg_diff, is_sub], axis=1)

print(f"Grupo E: {len(group_e):,} filas, {group_e.shape[1]} features, {time.time()-t0:.1f}s")
log_step('EXEC', f'Grupo E (competitividad): {group_e.shape[1]} features')

Grupo E: 0 filas, 3 features, 0.0s
[EXEC] 17:01:55 — Grupo E (competitividad): 3 features


In [17]:
# [EXEC] Grupo F: Boss fights (2 features)
t0 = time.time()

boss_fights = df_normalized[df_normalized['is_arena_boss'] == True]

if len(boss_fights) > 0:
    boss_count = boss_fights.groupby('user_id').size().rename('arena_boss_fights')
    boss_wr = boss_fights.groupby('user_id')['is_win'].mean().rename('arena_boss_win_rate').round(3)
    group_f = pd.concat([boss_count, boss_wr], axis=1)
else:
    group_f = pd.DataFrame(columns=['arena_boss_fights', 'arena_boss_win_rate'])

print(f"Grupo F: {len(group_f):,} filas, {group_f.shape[1]} features, {time.time()-t0:.1f}s")
log_step('EXEC', f'Grupo F (boss fights): {group_f.shape[1]} features')

Grupo F: 0 filas, 2 features, 0.0s
[EXEC] 17:01:55 — Grupo F (boss fights): 2 features


In [18]:
# [EXEC] Grupo G: Temporal (3 features)
t0 = time.time()

df_normalized['fight_date'] = df_normalized['fight_dt'].dt.date

group_g = df_normalized.groupby('user_id').agg(
    arena_first_fight=('fight_dt', 'min'),
    arena_last_fight=('fight_dt', 'max'),
    arena_days_active=('fight_date', 'nunique'),
)

df_normalized.drop(columns=['fight_date'], inplace=True)

print(f"Grupo G: {len(group_g):,} filas, {group_g.shape[1]} features, {time.time()-t0:.1f}s")
log_step('EXEC', f'Grupo G (temporal): {group_g.shape[1]} features')

Grupo G: 0 filas, 3 features, 0.0s
[EXEC] 17:01:55 — Grupo G (temporal): 3 features


In [19]:
# [EXEC] Combinar todos los grupos + reindex con sample_user_ids
t0 = time.time()

features = pd.concat([group_a, group_b, group_c, group_d, group_e, group_f, group_g], axis=1)

features = features.reindex(list(sample_user_ids))

# Fix B1: arena_has_rank = 1 si tenía ranking válido (rank>0 en algún combate)
# arena_rank_best = -1 como centinela para usuarios sin ranking válido.
features['arena_has_rank'] = features['arena_rank_best'].notna().astype(int)

date_cols = ['arena_first_fight', 'arena_last_fight']
numeric_cols = [c for c in features.columns if c not in date_cols]
features[numeric_cols] = features[numeric_cols].fillna(0)

# Sobrescribir arena_rank_best con -1 donde has_rank=0 (fillna(0) le habría puesto 0,
# pero 0 es una posición real válida — usamos -1 como centinela semántico)
features.loc[features['arena_has_rank'] == 0, 'arena_rank_best'] = -1

features = features.reset_index().rename(columns={'index': 'user_id'})

assert len(features) == N_SAMPLE, \
    f"ERROR: {len(features)} filas != {N_SAMPLE} sample"

print(f"Features finales: {features.shape}")
print(f"Verificación: {len(features)} filas == {N_SAMPLE} sample")
print(f"  arena_has_rank=1: {(features['arena_has_rank']==1).sum():,}")
print(f"  arena_has_rank=0 (centinela -1 en rank_best): {(features['arena_has_rank']==0).sum():,}")
log_step('EXEC', f'Features combinadas: {features.shape[0]:,} × {features.shape[1]} cols')
log_step('EXEC', f'Fix B1: arena_has_rank creado, arena_rank_best usa -1 como centinela')

Features finales: (25200, 23)
Verificación: 25200 filas == 25200 sample
  arena_has_rank=1: 0
  arena_has_rank=0 (centinela -1 en rank_best): 25,200
[EXEC] 17:01:55 — Features combinadas: 25,200 × 23 cols
[EXEC] 17:01:55 — Fix B1: arena_has_rank creado, arena_rank_best usa -1 como centinela


## 4. Validación de features

In [20]:
# [ANALYSIS] Tipos de datos finales + zeros vs nonzeros
print("TIPOS DE DATOS — FEATURES FINALES")
print("=" * 80)
for col in features.columns:
    dtype = features[col].dtype
    n_nulls = features[col].isnull().sum()
    if dtype in ['float64', 'int64', 'int32', 'float32']:
        n_zeros = (features[col] == 0).sum()
        n_nonzero = len(features) - n_nulls - n_zeros
        print(f"  {col:<35} dtype={str(dtype):<12} nulls={n_nulls:>8,}  "
              f"zeros={n_zeros:>8,}  nonzero={n_nonzero:>8,}")
    else:
        print(f"  {col:<35} dtype={str(dtype):<12} nulls={n_nulls:>8,}")

TIPOS DE DATOS — FEATURES FINALES
  user_id                             dtype=str          nulls=       0
  arena_fights_total                  dtype=float64      nulls=       0  zeros=  25,200  nonzero=       0
  arena_fights_as_attacker            dtype=float64      nulls=       0  zeros=  25,200  nonzero=       0
  arena_fights_as_defender            dtype=float64      nulls=       0  zeros=  25,200  nonzero=       0
  arena_wins_total                    dtype=float64      nulls=       0  zeros=  25,200  nonzero=       0
  arena_losses_total                  dtype=float64      nulls=       0  zeros=  25,200  nonzero=       0
  arena_win_rate                      dtype=float64      nulls=       0  zeros=  25,200  nonzero=       0
  arena_win_rate_as_attacker          dtype=float64      nulls=       0  zeros=  25,200  nonzero=       0
  arena_trophies_max                  dtype=float64      nulls=       0  zeros=  25,200  nonzero=       0
  arena_trophies_net                  dtype=fl

In [21]:
# [ANALYSIS] Nulos en features finales
nulls_f = features.isnull().sum().to_frame(name='n_nulls')
nulls_f['pct_nulls'] = (nulls_f['n_nulls'] / len(features) * 100).round(2)
nulls_f = nulls_f[nulls_f['n_nulls'] > 0].sort_values('pct_nulls', ascending=False)

if len(nulls_f) > 0:
    print("Columnas con nulos (esperado solo en fechas):")
    print(nulls_f)
else:
    print("No hay nulos")

nulls_f.to_csv(REPORT_DIR / 'nulos_features_final.csv')
log_step('ANALYSIS', f'Nulos: {len(nulls_f)} columnas con nulos')

Columnas con nulos (esperado solo en fechas):
                   n_nulls  pct_nulls
arena_first_fight    25200      100.0
arena_last_fight     25200      100.0
[ANALYSIS] 17:01:55 — Nulos: 2 columnas con nulos


In [22]:
# [ANALYSIS] Estadísticas descriptivas e histogramas
numeric_features = features.select_dtypes(include=[np.number])
desc = numeric_features.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).T.round(2)
print(desc)
desc.to_csv(REPORT_DIR / 'features_describe.csv')

key_features = [
    'arena_fights_total', 'arena_win_rate', 'arena_trophies_max',
    'arena_gold_earned', 'arena_category_max', 'arena_avg_level_diff',
    'arena_boss_fights', 'arena_days_active', 'arena_rank_best',
]

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, feat in zip(axes.flatten(), key_features):
    if feat in features.columns:
        data = features[feat][features[feat] > 0]
        if len(data) > 0:
            ax.hist(np.log1p(data), bins=50, color='steelblue', alpha=0.7)
            ax.set_title(f'{feat} (log1p, n={len(data):,})')
        else:
            ax.set_title(f'{feat} (sin datos)')

plt.tight_layout()
plt.savefig(REPORT_DIR / 'features_distributions.png', dpi=100, bbox_inches='tight')
plt.close()
log_step('ANALYSIS', 'Estadísticas y histogramas guardados')

                              count  mean  std  min  25%  50%  75%  90%  99%  max
arena_fights_total          25200.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
arena_fights_as_attacker    25200.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
arena_fights_as_defender    25200.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
arena_wins_total            25200.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
arena_losses_total          25200.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
arena_win_rate              25200.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
arena_win_rate_as_attacker  25200.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
arena_trophies_max          25200.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
arena_trophies_net          25200.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
arena_rank_best             25200.0  -1.0  0.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0
arena_gold_earned           25200.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
arena_exp_earned

[ANALYSIS] 17:01:55 — Estadísticas y histogramas guardados


In [23]:
# [EXEC] Guardar features_arena_cutoff.parquet
features.to_parquet(PARQUET_FEATURES, index=False, compression='snappy')
size_mb = PARQUET_FEATURES.stat().st_size / 1e6
log_step('EXEC', f'features_arena_cutoff.parquet: {features.shape}, {size_mb:.1f} MB')
print(f"Guardado: {PARQUET_FEATURES} ({size_mb:.1f} MB)")

[EXEC] 17:01:55 — features_arena_cutoff.parquet: (25200, 23), 0.6 MB
Guardado: /Users/jezquerro/Documents/tfg/data/data_qc/features_arena_cutoff.parquet (0.6 MB)


In [24]:
# [EXEC] Guardar muestra y liberar memoria
features.head(20).to_csv(REPORT_DIR / 'sample_head.csv', index=False)
log_step('EXEC', 'sample_head.csv guardado')

del df_normalized
gc.collect()
print("Memoria liberada")

[EXEC] 17:01:55 — sample_head.csv guardado
Memoria liberada


## 5. Informe de ejecución

In [25]:
# [REPORT] Generar execution_report.md
t_total = time.time() - NOTEBOOK_START
t_min = int(t_total // 60)
t_sec = int(t_total % 60)
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

n_con_arena = features[features['arena_fights_total'] > 0].shape[0]
n_sin_arena = features[features['arena_fights_total'] == 0].shape[0]

lines = []
lines += [
    "# Informe de ejecucion — arena_log.csv",
    "",
    f"**Notebook:** notebooks/fase1_cleaning/02d_arena_log.ipynb",
    f"**Fecha:** {now_str}",
    f"**Tiempo de ejecucion:** {t_min} min {t_sec}s",
    f"**CSV de entrada:** data/data_raw/arena_log.csv (71 MB, {n_before:,} filas, 30 cols)",
    f"**Parquet de salida:** data/data_qc/features_arena_cutoff.parquet ({features.shape[0]:,} × {features.shape[1]} cols)",
    "",
    "---", "",
    "## Hallazgo crítico: character_ids en arena_log",
    "",
    "Los campos `attacker_id`, `defender_id` y `winner_id` de arena_log.csv",
    "contienen **character_ids** (ObjectId del personaje), NO user_ids.",
    "Esto se descubrió en la primera iteración cuando el filtrado por",
    "sample_user_ids daba 0 matches.",
    "",
    "**Solución:** se construye un mapeo `char_id → user_id` desde",
    f"`characters.csv` ({len(char_to_user):,} entradas) y se traducen los IDs",
    "antes de filtrar.",
    "",
    "**Advertencia:** otras tablas como `fights_log.csv` y `arena_global_ranking.csv`",
    "probablemente usen el mismo patrón (character_ids en vez de user_ids).",
    "Verificar en cada notebook antes de filtrar.",
    "",
    "---", "",
    "## Resumen ejecutivo",
    "",
    f"Se procesaron {n_before:,} combates de arena. Tras traducir character_ids",
    f"a user_ids y filtrar por el sample, quedaron {n_after:,} combates relevantes.",
    f"Se normalizaron a la perspectiva del usuario y se generaron",
    f"{features.shape[1] - 1} features en 7 grupos.",
    "",
    f"- Usuarios con combates de arena: {n_con_arena:,} ({n_con_arena/len(features)*100:.1f}%)",
    f"- Usuarios sin combates de arena: {n_sin_arena:,} ({n_sin_arena/len(features)*100:.1f}%)",
    "",
    "---", "",
    "## Complejidad: doble columna + traducción de IDs",
    "",
    "Cada combate involucra dos jugadores: `attacker_id` y `defender_id`.",
    "Un usuario puede aparecer como atacante en unos combates y defensor en otros.",
    "",
    "Además, estos IDs son character_ids, no user_ids. El pipeline es:",
    "1. Construir mapeo char_id → user_id desde characters.csv",
    "2. Traducir attacker_id/defender_id a user_ids",
    "3. Filtrar combates donde al menos un jugador está en el sample",
    "4. Normalizar a perspectiva del usuario (attacker/defender → user_id unificado)",
    "5. Calcular is_win comparando char_ids originales con winner_id (también char_id)",
    "",
    "---", "",
    "## Constantes",
    "",
    "| Constante | Valor |",
    "|---|---|",
    f"| `REFERENCE_DATE` | {REFERENCE_DATE} |",
    f"| `N_SAMPLE` | {N_SAMPLE:,} |",
    f"| `char_to_user` | {len(char_to_user):,} entradas |",
    "",
    "---", "",
    "## Pasos ejecutados",
    "",
]
for s in steps_log:
    lines.append(f"- {s}")

lines += [
    "",
    "---", "",
    "## Filtrado y normalización",
    "",
    "| Paso | Filas |",
    "|---|---|",
    f"| CSV original | {n_before:,} |",
    f"| Combates con jugador del sample | {n_after:,} |",
    "",
    "---", "",
    f"## Features generadas ({features.shape[1] - 1} + user_id)",
    "",
    "### Grupo A — Volumen (3)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `arena_fights_total` | Total combates |",
    "| `arena_fights_as_attacker` | Combates como atacante |",
    "| `arena_fights_as_defender` | Combates como defensor |",
    "",
    "### Grupo B — Rendimiento (4)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `arena_wins_total` | Victorias totales |",
    "| `arena_losses_total` | Derrotas totales |",
    "| `arena_win_rate` | wins / total (0-1) |",
    "| `arena_win_rate_as_attacker` | Win rate como atacante |",
    "",
    "### Grupo C — Trofeos (3)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `arena_trophies_max` | Máximo trofeos alcanzado |",
    "| `arena_trophies_net` | Suma neta trofeos |",
    "| `arena_rank_best` | Mejor ranking (min) |",
    "",
    "### Grupo D — Recompensas (3)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `arena_gold_earned` | Gold ganado |",
    "| `arena_exp_earned` | Experiencia ganada |",
    "| `arena_dark_steel_earned` | Dark steel ganado |",
    "",
    "### Grupo E — Competitividad (3)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `arena_category_max` | Máxima categoría |",
    "| `arena_avg_level_diff` | Dif. media nivel vs oponente |",
    "| `arena_is_subscriber` | 1 si fue suscriptor |",
    "",
    "### Grupo F — Boss fights (2)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `arena_boss_fights` | Combates contra bosses |",
    "| `arena_boss_win_rate` | Win rate contra bosses |",
    "",
    "### Grupo G — Temporal (3)",
    "| Feature | Descripción |",
    "|---|---|",
    "| `arena_first_fight` | Fecha primer combate |",
    "| `arena_last_fight` | Fecha último combate |",
    "| `arena_days_active` | Días con combates |",
    "",
    "---", "",
    "## Lo que ha ido bien",
    "",
    "- Mapeo char_id → user_id funciona correctamente",
    "- Normalización attacker/defender correcta",
    "- is_win calculado con char_ids originales (no traducidos)",
    "- Recompensas solo asignadas al atacante",
    "- Trofeos del defensor invertidos correctamente",
    "",
    "---", "",
    "## Lo que ha ido mal o requirió ajustes",
    "",
    "**Iteración 1:** 0 matches al filtrar por sample_user_ids.",
    "  - **Causa:** attacker_id/defender_id son character_ids, no user_ids.",
    "  - **Solución:** construir mapeo char_id → user_id desde characters.csv.",
    "  - **Impacto:** requirió cargar characters.csv adicional (solo _id y user_id).",
    "",
    "---", "",
    "## Decisiones tomadas",
    "",
    "- Mapeo char_id → user_id construido desde characters.csv",
    "- is_win calculado comparando char_ids originales (attacker_id/defender_id vs winner_id)",
    "- Recompensas solo sumadas para atacantes (defensores → 0)",
    "- trophies_change invertido para defensores (-abs)",
    "- arena_rank_best = min(rank) porque rank menor = mejor posición",
    "- Boss fights separados como grupo (~4% del total)",
    "",
    "---", "",
]

feat_size = PARQUET_FEATURES.stat().st_size / 1e6
lines += [
    "## Archivos generados",
    "",
    "| Archivo | Tamaño |",
    "|---|---|",
    f"| features_arena_cutoff.parquet ({features.shape[1]} cols) | {feat_size:.1f} MB |",
    "| nulos_por_columna_raw.csv | - |",
    "| nulos_features_final.csv | - |",
    "| features_describe.csv | - |",
    "| features_distributions.png | - |",
    "| sample_head.csv | - |",
    "| execution_report.md | - |",
    "",
    "---", "",
    "## Advertencias para notebooks siguientes",
    "",
    "- **arena_log usa character_ids, no user_ids** — construir mapeo primero",
    "- **fights_log probablemente usa el mismo patrón** — verificar",
    "- Usar `user_id` (hex 24 chars) para joins con otros parquets",
    f"- REFERENCE_DATE = {REFERENCE_DATE}",
    f"- ~{n_sin_arena/len(features)*100:.0f}% de usuarios no tienen arena (features a cero)",
    "",
    "---", "",
    "## Tareas pendientes",
    "",
    "- [ ] Verificar si fights_log usa character_ids (probable)",
    "- [ ] Investigar correlación arena_win_rate vs churn",
    "- [ ] Considerar feature de tendencia (win rate reciente vs histórico)",
    "- [ ] Evaluar arena_category_max como ordinal o one-hot en 02z",
    "",
    "---", "",
    "## Diagrama del pipeline",
    "",
    "```",
    "characters.csv (_id, user_id)",
    "    │",
    "    ├─ Extraer char_id desde ObjectId(_id)",
    f"    ├─ Construir mapeo char_id → user_id ({len(char_to_user):,} entradas)",
    "    ▼",
    f"arena_log.csv ({n_before:,} combates, 30 cols)",
    "    │",
    "    ├─ Traducir attacker_id/defender_id → user_ids (via mapeo)",
    f"    ├─ Filtrar: attacker_user_id OR defender_user_id in sample  (-{n_before - n_after:,})",
    "    ▼",
    f"Combates filtrados ({n_after:,})",
    "    │",
    "    ├─ Normalizar attacker → user_id + role='attacker'",
    "    ├─ Normalizar defender → user_id + role='defender'",
    "    ├─ is_win = (char_id_original == winner_id)",
    "    ▼",
    "df_normalized",
    "    │",
    "    ├─ Grupos A-G de features",
    "    ├─ Reindex con sample_user_ids",
    "    ▼",
    f"features_arena_cutoff.parquet ({features.shape[0]:,} × {features.shape[1]})",
    "```",
    "",
    "---", "",
    "## Reproducibilidad",
    "",
    "1. Ejecutar 02a_users.ipynb primero (genera sample_user_ids)",
    "2. Ejecutar 02d_arena_log.ipynb (necesita characters.csv para mapeo)",
    f"3. Verificar: features_arena_cutoff.parquet == {N_SAMPLE:,} filas",
    "",
]

report_path = REPORT_DIR / 'execution_report.md'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
print(f"Informe guardado: {report_path}")
log_step('REPORT', 'execution_report.md generado')

Informe guardado: /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/arena_log/execution_report.md
[REPORT] 17:01:56 — execution_report.md generado


In [26]:
# [REPORT] Resumen final
elapsed = time.time() - NOTEBOOK_START

print("=" * 70)
print("RESUMEN FINAL — Notebook 02d_arena_log")
print("=" * 70)
print(f"  Tiempo total          : {int(elapsed//60)}m {int(elapsed%60)}s")
print(f"  Combates CSV original : {n_before:,}")
print(f"  Combates filtrados    : {n_after:,}")
print(f"  Mapeo char→user       : {len(char_to_user):,} entradas")
print(f"  Usuarios con arena    : {n_users_arena:,} ({n_users_arena/N_SAMPLE*100:.1f}%)")
print(f"  Usuarios sin arena    : {n_users_no_arena:,} ({n_users_no_arena/N_SAMPLE*100:.1f}%)")
print(f"  Filas features final  : {len(features):,} (== {N_SAMPLE:,} sample)")
print(f"  Columnas features     : {features.shape[1]}")
print()
print("Archivos generados:")
print(f"  features_arena_cutoff.parquet ({PARQUET_FEATURES.stat().st_size/1e6:.1f} MB)")
print(f"  execution_report.md")
print(f"  Gráficos y CSVs en {REPORT_DIR}")
print("=" * 70)
print("Siguiente paso: revisar execution_report.md y compartirlo con Claude")

RESUMEN FINAL — Notebook 02d_arena_log
  Tiempo total          : 0m 8s
  Combates CSV original : 249,583
  Combates filtrados    : 116,933
  Mapeo char→user       : 880,095 entradas
  Usuarios con arena    : 0 (0.0%)
  Usuarios sin arena    : 25,200 (100.0%)
  Filas features final  : 25,200 (== 25,200 sample)
  Columnas features     : 23

Archivos generados:
  features_arena_cutoff.parquet (0.6 MB)
  execution_report.md
  Gráficos y CSVs en /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/arena_log
Siguiente paso: revisar execution_report.md y compartirlo con Claude


In [27]:
# [REPORT] Logging dual: HTML + sección enriquecida del report
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from notebook_logging_template import (
    export_notebook_to_html, build_enriched_report_section,
)

notebook_path = PROJECT_ROOT / 'notebooks' / 'fase1_cleaning' / '02d_arena_log.ipynb'
html_path = REPORT_DIR / '02d_arena_log_run.html'
export_notebook_to_html(notebook_path, html_path)

enriched = build_enriched_report_section(
    df_final=features,
    raw_shape=(249583, 30),
)
with open(REPORT_DIR / 'execution_report.md', 'a', encoding='utf-8') as f:
    f.write('\n\n---\n\n' + enriched)
print(f"Enriquecimiento appendeado a {REPORT_DIR / 'execution_report.md'}")

HTML generado: 02d_arena_log_run.html (0.5 MB)
Enriquecimiento appendeado a /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/arena_log/execution_report.md
